# LSTM 对照实验（LSTM_Funning）

## 目的
使用LSTM（长短期记忆网络）模型进行文本分类实验，作为BERT模型的对照基线。使用与LoRA实验一致的数据划分与超参数，输出统一指标格式。

## 数据流向
输入：原始数据文件（waimai.csv）-> 构建词汇表 -> 文本编码 -> LSTM模型训练 -> 模型评估
输出：训练好的LSTM模型、训练过程指标、测试集评估结果

## 操作步骤
1. 导入必要的库和配置
2. 从训练数据构建字符级词汇表
3. 将文本编码为ID序列
4. 定义LSTM分类模型
5. 准备训练数据
6. 执行训练循环
7. 在测试集上评估模型性能

In [1]:
"""
目的：导入LSTM实验所需的所有Python库和模块

数据流向：
输入：无（直接导入库）
输出：所有必要的库和函数已加载到当前命名空间

操作步骤：
1. 导入系统库：os（文件操作）、time（时间计算）
2. 导入数据结构：Counter（用于统计字符频率）
3. 导入PyTorch相关：torch（张量计算）、nn（神经网络模块）、Dataset（数据集基类）、DataLoader（数据加载）
4. 导入评估指标：sklearn的准确率、精确率、召回率、F1分数计算函数
5. 导入进度条：tqdm（显示训练进度）
6. 导入自定义配置：从Bert_Config导入统一配置和工具函数（数据加载、分割、保存路径）
"""

import os
import time
from collections import Counter
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tqdm import tqdm

from Bert_Config import CONFIG, setup_seed, load_raw_data, split_data, get_save_path

print("✅ 所有库导入完成")


C:\Users\19836\miniconda3\envs\py8\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ 所有库导入完成


In [2]:
"""
目的：从统一配置文件加载实验参数，设置LSTM模型的特定配置参数

数据流向：
输入：CONFIG字典（从Bert_Config导入）-> 提取配置参数 -> 设置LSTM特定参数
输出：所有配置变量已设置，配置信息已打印

操作步骤：
1. 从CONFIG字典提取通用配置：数据路径、随机种子、最大文本长度、训练轮数、批次大小
2. 设置LSTM模型特定参数：
   - MAX_VOCAB=7000：词汇表最大容量（字符级）
   - EMBED_DIM=256：词嵌入维度
   - HIDDEN_DIM=256：LSTM隐藏层维度
   - NUM_LAYERS=2：LSTM层数（多层可以提取更复杂的特征）
   - BIDIRECTIONAL=True：使用双向LSTM（同时利用前后文信息）
   - LEARNING_RATE=5e-4：学习率
3. 获取模型保存路径：调用get_save_path("lstm")获取保存目录
4. 设置实验名称：EXP_NAME = "LSTM对照实验"
5. 打印所有配置参数，便于确认实验设置
"""

MODEL_NAME = "LSTM"
DATA_PATH = CONFIG["DATA_DIR"]
RANDOM_SEED = int(os.environ.get("NOTEBOOK_SEED", CONFIG["RANDOM_SEED"]))

MAX_LEN = CONFIG["CLASSIC_MAX_LENGTH_TOKENS"]
EPOCHS = CONFIG["CLASSIC_EPOCHS"]
BATCH_SIZE = CONFIG["CLASSIC_BATCH_SIZE"]

MAX_VOCAB = 7000
EMBED_DIM = 256
HIDDEN_DIM = 256
NUM_LAYERS = 2
BIDIRECTIONAL = True
LEARNING_RATE = 5e-4

SAVE_PATH = get_save_path("lstm")

EXP_NAME = "LSTM对照实验"
print("=" * 50)
print(f"实验配置: {EXP_NAME}")
print("=" * 50)
print(f"模型名称: {MODEL_NAME}")
print(f"数据目录: {DATA_PATH}")
print(f"随机种子: {RANDOM_SEED}")
print(f"最大词汇量: {MAX_VOCAB}")
print(f"最大文本长度: {MAX_LEN}")
print(f"词嵌入维度: {EMBED_DIM}")
print(f"隐藏层维度: {HIDDEN_DIM}")
print(f"LSTM层数: {NUM_LAYERS}")
print(f"是否双向: {BIDIRECTIONAL}")
print(f"训练轮数: {EPOCHS}")
print(f"批次大小: {BATCH_SIZE}")
print(f"学习率: {LEARNING_RATE}")
print(f"模型保存路径: {SAVE_PATH}")
print("=" * 50)


实验配置: LSTM对照实验
模型名称: LSTM
数据目录: ../hotel.csv
随机种子: 40
最大词汇量: 7000
最大文本长度: 128
词嵌入维度: 256
隐藏层维度: 256
LSTM层数: 2
是否双向: True
训练轮数: 5
批次大小: 64
学习率: 0.0005
模型保存路径: ./lstm_checkpoint


In [ ]:
"""
目的：定义文本编码函数和数据集类，用于将原始文本转换为模型可处理的数字序列

数据流向：
输入：原始文本列表 -> 构建词汇表 -> 文本编码 -> 数据集对象
输出：TextDataset对象（可以迭代获取编码后的文本和标签）

操作步骤：
1. 定义build_vocab函数：从训练文本构建字符级词汇表
2. 定义encode_text函数：将文本转换为固定长度的ID序列
3. 定义TextDataset类：封装文本和标签，支持PyTorch数据加载
"""

def build_vocab(texts, max_vocab):
    """
    目的：从训练文本中构建字符级词汇表，将字符映射为数字ID
    
    输入：
      - texts: 文本列表（通常是训练集的文本）
      - max_vocab: 词汇表的最大容量
    输出：
      - vocab: 字典，字符到ID的映射
    
    数据流向：
    输入文本列表 -> 统计字符频率 -> 选择高频字符 -> 构建词汇字典
    
    操作步骤：
    1. 使用Counter统计所有文本中每个字符的出现频率
    2. 选择出现频率最高的 max_vocab-2 个字符（保留2个位置给特殊标记）
    3. 创建词汇表字典，包含两个特殊标记：
       - "<PAD>": 0 (填充标记，用于补齐短文本)
       - "<UNK>": 1 (未知字符标记，用于处理未在词汇表中的字符)
    4. 将高频字符按频率从高到低依次分配ID（从2开始）
    5. 返回完整的词汇表字典
    """
    counter = Counter()
    for text in texts:
        counter.update(list(text))
    most_common = counter.most_common(max_vocab - 2)
    vocab = {"<PAD>": 0, "<UNK>": 1}
    for idx, (tok, _) in enumerate(most_common, start=2):
        vocab[tok] = idx
    return vocab


def encode_text(text, vocab, max_len):
    """
    目的：将原始文本转换为固定长度的数字ID序列，便于模型处理
    
    输入：
      - text: 原始文本字符串
      - vocab: 词汇表字典（字符到ID的映射）
      - max_len: 目标序列长度
    输出：
      - ids: 整数列表，长度为max_len
    
    数据流向：
    输入文本字符串 -> 字符列表 -> ID列表 -> 填充/截断 -> 固定长度ID序列
    
    操作步骤：
    1. 将文本字符串转换为字符列表
    2. 截取前max_len个字符（如果文本过长）
    3. 将每个字符转换为对应的ID：
       - 如果字符在词汇表中，使用其ID
       - 如果字符不在词汇表中，使用"<UNK>"的ID（1）
    4. 如果ID序列长度小于max_len，在末尾添加"<PAD>"的ID（0）进行填充
    5. 返回固定长度的ID序列
    """
    tokens = list(text)
    ids = [vocab.get(tok, vocab["<UNK>"]) for tok in tokens[:max_len]]
    if len(ids) < max_len:
        ids.extend([vocab["<PAD>"]] * (max_len - len(ids)))
    return ids


class TextDataset(Dataset):
    """
    目的：将文本数据和标签封装为PyTorch数据集，支持批量加载和迭代
    
    数据流向：
    输入：文本列表、标签列表、词汇表、最大长度
    输出：每次迭代返回 (编码后的文本张量, 标签张量)
    """
    
    def __init__(self, texts, labels, vocab, max_len):
        """
        目的：初始化数据集，存储文本、标签和编码所需参数
        
        输入：
          - texts: 文本列表
          - labels: 标签列表
          - vocab: 词汇表（用于编码）
          - max_len: 最大长度（用于统一文本长度）
        输出：无（保存为实例属性）
        
        数据流向：
        输入参数 -> 保存为实例属性
        
        操作步骤：
        1. 保存文本列表
        2. 保存标签列表
        3. 保存词汇表（用于编码）
        4. 保存最大长度（用于统一文本长度）
        """
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        """
        目的：返回数据集的大小，供DataLoader使用
        
        输入：无
        输出：数据集样本数量（整数）
        
        数据流向：
        无输入 -> 返回标签列表长度
        
        操作步骤：
        直接返回标签列表的长度（每个标签对应一个样本）
        """
        return len(self.labels)

    def __getitem__(self, idx):
        """
        目的：根据索引获取单个样本，将文本编码为张量，标签转换为张量
        
        输入：
          - idx: 样本索引
        输出：
          - (input_ids张量, label张量)
        
        数据流向：
        输入索引 -> 获取文本和标签 -> 编码文本 -> 转换为张量
        
        操作步骤：
        1. 根据索引idx获取对应的文本和标签
        2. 使用encode_text函数将文本转换为ID序列
        3. 将ID序列转换为PyTorch长整型张量
        4. 将标签转换为PyTorch长整型张量
        5. 返回文本张量和标签张量的元组
        """
        input_ids = encode_text(self.texts[idx], self.vocab, self.max_len)
        return torch.tensor(input_ids, dtype=torch.long), torch.tensor(
            self.labels[idx], dtype=torch.long
        )

print("✅ 数据集构建函数定义完成")


In [ ]:
"""
目的：定义LSTM文本分类模型类，包含词嵌入层、双向多层LSTM层和分类层

数据流向：
输入：文本ID序列 (batch_size, seq_len) 
-> 词嵌入层 -> 双向多层LSTM层 -> 全连接层 
-> 输出：类别logits (batch_size, num_classes)

操作步骤：
1. 定义LSTMClassifier类，继承自nn.Module
2. 在__init__中初始化各层组件
3. 在forward中定义前向传播过程
"""

class LSTMClassifier(nn.Module):
    """
    目的：定义LSTM文本分类模型，用于对文本进行情感分类（二分类）
    
    输入：
      - vocab_size: 词汇表大小
      - embed_dim: 词向量维度
      - hidden_dim: LSTM隐藏层维度
      - num_layers: LSTM层数（默认2层）
      - bidirectional: 是否使用双向LSTM（默认True）
      - num_classes: 分类类别数（默认2）
    输出：模型对象
    
    数据流向：
    输入参数 -> 初始化各层组件 -> 返回模型对象
    
    操作步骤：
    1. 调用父类初始化方法
    2. 创建词嵌入层：将字符ID映射为embed_dim维的向量
       - padding_idx=0 表示ID为0（填充标记）的向量始终为0
    3. 创建多层双向LSTM层：处理序列信息
       - num_layers: LSTM层数，多层可以提取更复杂的特征
       - bidirectional: 双向LSTM能同时利用前后文信息
       - batch_first=True 表示输入格式为 (batch, seq, feature)
       - dropout: 多层时在层间添加dropout防止过拟合
    4. 创建Dropout层：防止过拟合，随机丢弃20%的神经元
    5. 创建全连接层：将LSTM输出映射到类别数
       - 双向LSTM时输入维度为 hidden_dim * 2
    """
    
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers=2, bidirectional=True, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim, 
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=0.2 if num_layers > 1 else 0
        )
        self.dropout = nn.Dropout(0.2)
        fc_input_dim = hidden_dim * 2 if bidirectional else hidden_dim
        self.fc = nn.Linear(fc_input_dim, num_classes)

    def forward(self, x):
        """
        目的：定义前向传播过程，将输入文本转换为分类结果
        
        输入：
          - x: (batch_size, seq_len) - 文本ID序列
        输出：
          - logits: (batch_size, num_classes) - 类别logits（未经过softmax）
        
        数据流向：
        输入ID序列 -> embedding -> LSTM -> 取最后时刻隐藏状态 -> dropout -> 全连接层 -> 输出logits
        
        操作步骤：
        1. 通过词嵌入层将ID序列转换为词向量序列 (batch_size, seq_len, embed_dim)
        2. 通过双向多层LSTM层处理序列，获取每个时间步的隐藏状态
        3. 取LSTM最后一层的最后一个时间步的隐藏状态（代表整个序列的信息）
           - 双向LSTM时，hidden的形状为 (num_layers * 2, batch_size, hidden_dim)
           - 拼接前向和后向的最后一个隐藏状态 (batch_size, hidden_dim * 2)
        4. 应用Dropout进行正则化，防止过拟合
        5. 通过全连接层将隐藏状态映射到类别数，得到分类logits
        6. 返回分类结果（未经过softmax，用于计算交叉熵损失）
        """
        x = self.embedding(x)
        _, (hidden, _) = self.lstm(x)
        # 双向LSTM时，hidden的形状为 (num_layers * 2, batch_size, hidden_dim)
        # 取最后一层（包含前向和后向）
        if self.lstm.bidirectional:
            # 拼接前向和后向的最后一个隐藏状态
            out = torch.cat([hidden[-2], hidden[-1]], dim=1)
        else:
            out = hidden[-1]
        out = self.dropout(out)
        return self.fc(out)

print("✅ 模型定义完成")

In [ ]:

"""
目的：定义模型评估函数，用于在验证集和测试集上评估模型性能

数据流向：
输入：model（模型）、dataloader（数据加载器）、criterion（损失函数）、device（设备）
-> 批量评估 -> 收集预测和标签 -> 计算指标
输出：评估指标元组（准确率、精确率、召回率、F1分数、平均损失）

操作步骤：
1. 定义evaluate函数，实现模型评估逻辑
"""

def evaluate(model, dataloader, criterion, device):
    """
    目的：评估模型在数据集上的性能，计算准确率、精确率、召回率、F1分数和平均损失
    
    输入：
      - model: 训练好的模型
      - dataloader: 数据加载器（验证集或测试集）
      - criterion: 损失函数
      - device: 计算设备（CPU或GPU）
    输出：
      - acc: 准确率（float）
      - precision: 精确率（float）
      - recall: 召回率（float）
      - f1: F1分数（float）
      - avg_loss: 平均损失（float）
    
    数据流向：
    输入数据批次 -> 模型前向传播 -> 计算损失和预测 -> 收集所有结果 -> 计算指标
    
    操作步骤：
    1. 将模型设置为评估模式（关闭dropout等训练时的特殊行为）
    2. 初始化累计损失和标签/预测列表
    3. 关闭梯度计算（节省内存和计算资源）
    4. 遍历数据加载器中的每个批次：
       a. 将数据移动到指定设备（CPU或GPU）
       b. 通过模型前向传播得到预测logits
       c. 计算批次损失并累加（乘以批次大小以计算总损失）
       d. 通过argmax获取预测类别（概率最大的类别）
       e. 将真实标签和预测标签收集到列表中
    5. 计算平均损失（总损失除以样本总数）
    6. 使用sklearn计算准确率、精确率、召回率、F1分数（二分类）
    7. 返回所有评估指标
    """
    model.eval()
    total_loss = 0.0
    all_labels = []
    all_preds = []
    with torch.no_grad():
        for batch_x, batch_y in dataloader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            total_loss += loss.item() * batch_y.size(0)
            preds = logits.argmax(dim=1)
            all_labels.extend(batch_y.cpu().numpy().tolist())
            all_preds.extend(preds.cpu().numpy().tolist())

    avg_loss = total_loss / len(dataloader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="binary", zero_division=0
    )
    return acc, precision, recall, f1, avg_loss

print("✅ 评估函数定义完成")


In [ ]:
"""
目的：加载原始数据，构建词汇表，创建训练集、验证集和测试集的数据集对象和数据加载器

数据流向：
输入：DATA_PATH（数据文件路径）、RANDOM_SEED（随机种子）、MAX_VOCAB（最大词汇量）、MAX_LEN（最大长度）
-> 加载数据 -> 划分数据集 -> 构建词汇表 -> 创建数据集对象 -> 创建数据加载器
输出：train_loader（训练数据加载器）、val_loader（验证数据加载器）、test_loader（测试数据加载器）

操作步骤：
1. 设置随机种子，确保数据划分可复现
2. 调用load_raw_data加载原始CSV数据
3. 调用split_data将数据按8:1:1划分为训练集、验证集、测试集
4. 使用训练集文本构建词汇表：调用build_vocab函数，传入训练集文本和最大词汇量
5. 创建训练数据集：使用TextDataset封装训练集文本、标签、词汇表和最大长度
6. 创建验证数据集：使用TextDataset封装验证集文本、标签、词汇表和最大长度
7. 创建测试数据集：使用TextDataset封装测试集文本、标签、词汇表和最大长度
8. 创建训练数据加载器：使用DataLoader封装训练集，设置批次大小和shuffle=True
9. 创建验证数据加载器：使用DataLoader封装验证集，设置批次大小，不shuffle
10. 创建测试数据加载器：使用DataLoader封装测试集，设置批次大小，不shuffle
11. 打印数据准备完成提示
"""

setup_seed(RANDOM_SEED)
df = load_raw_data(DATA_PATH)
train_df, val_df, test_df = split_data(df, RANDOM_SEED)

vocab = build_vocab(train_df["review"], MAX_VOCAB)

train_dataset = TextDataset(
    train_df["review"].tolist(),
    train_df["label"].astype(int).tolist(),
    vocab,
    MAX_LEN,
)
val_dataset = TextDataset(
    val_df["review"].tolist(),
    val_df["label"].astype(int).tolist(),
    vocab,
    MAX_LEN,
)
test_dataset = TextDataset(
    test_df["review"].tolist(),
    test_df["label"].astype(int).tolist(),
    vocab,
    MAX_LEN,
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

print("✅ 数据准备完成")


In [ ]:
"""
目的：定义模型保存函数，用于将训练好的模型参数保存到磁盘

数据流向：
输入：model（模型对象）、save_name（保存的文件名）
-> 检查目录 -> 保存模型参数
输出：无（模型参数保存到文件）

操作步骤：
1. 定义save_model函数，实现模型保存逻辑
"""

def save_model(model, save_name):
    """
    目的：将训练好的模型参数保存到磁盘，便于后续加载和使用
    
    输入：
      - model: 待保存的模型对象
      - save_name: 保存的文件名（如"best.pt"或"last.pt"）
    输出：无（模型参数保存到文件）
    
    数据流向：
    模型参数 -> 提取state_dict -> 保存到文件
    
    操作步骤：
    1. 检查保存目录是否存在，如果不存在则创建
    2. 拼接完整的保存路径（目录 + 文件名）
    3. 使用torch.save保存模型的state_dict（模型参数字典）
    4. 打印保存成功的消息和文件路径
    """
    if not os.path.exists(SAVE_PATH):
        os.makedirs(SAVE_PATH)
    save_file = os.path.join(SAVE_PATH, save_name)
    torch.save(model.state_dict(), save_file)
    print(f"✅ LSTM模型已保存: {save_file}")

print("✅ 模型保存函数定义完成")


In [ ]:
"""
目的：初始化模型、损失函数、优化器和学习率调度器，执行LSTM模型训练循环

数据流向：
输入：vocab（词汇表）、train_loader（训练数据）、val_loader（验证数据）
-> 初始化模型和训练环境 -> 训练循环 -> 每个epoch：训练批次 -> 验证评估 -> 保存最佳模型
输出：训练好的模型（保存为best.pt和last.pt）、训练指标、训练时间

操作步骤：
1. 检测并设置计算设备（CPU或GPU）
2. 创建LSTM模型实例，传入词汇表大小、嵌入维度、隐藏层维度、层数、是否双向等参数
3. 将模型移动到指定设备
4. 创建交叉熵损失函数
5. 创建Adam优化器，传入模型参数和学习率
6. 创建学习率调度器（ReduceLROnPlateau）：当验证准确率不再提升时降低学习率
7. 初始化训练开始时间、最佳验证准确率
8. 对每个epoch执行：
   a. 设置模型为训练模式
   b. 初始化训练准确率和损失累计器
   c. 遍历训练数据批次：
      - 将数据移动到设备
      - 前向传播得到logits
      - 计算损失
      - 反向传播计算梯度
      - 梯度裁剪（防止梯度爆炸）
      - 优化器更新参数
   d. 在验证集上评估模型性能
   e. 计算训练集平均损失和准确率
   f. 打印epoch训练和验证指标
   g. 根据验证准确率更新学习率
   h. 如果验证准确率提升，保存最佳模型
9. 训练结束后保存最后一个epoch的模型
10. 打印训练完成信息和模型保存路径
"""

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LSTMClassifier(
    len(vocab), 
    EMBED_DIM, 
    HIDDEN_DIM, 
    num_layers=NUM_LAYERS,
    bidirectional=BIDIRECTIONAL
).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=2, verbose=True
)

print("\n开始训练...")
train_start_time = time.time()
best_dev_acc = 0

for epoch_num in range(EPOCHS):
    model.train()
    total_acc_train = 0
    total_loss_train = 0

    for batch_x, batch_y in tqdm(
        train_loader,
        desc=f"Epoch {epoch_num + 1}/{EPOCHS} [训练]",
    ):
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        logits = model(batch_x)
        batch_loss = criterion(logits, batch_y)
        total_loss_train += batch_loss.item() * batch_y.size(0)

        acc = (logits.argmax(dim=1) == batch_y).sum().item()
        total_acc_train += acc

        optimizer.zero_grad()
        batch_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

    val_acc, val_precision, val_recall, val_f1, val_loss = evaluate(
        model, val_loader, criterion, device
    )
    
    train_loss_avg = total_loss_train / len(train_dataset)
    train_acc_avg = total_acc_train / len(train_dataset)
    
    print(
        f"[Epoch {epoch_num + 1}/{EPOCHS}] "
        f"Train Loss: {train_loss_avg:.4f} | "
        f"Train Acc: {train_acc_avg:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val F1: {val_f1:.4f}"
    )

    scheduler.step(val_acc)
    
    if val_acc > best_dev_acc:
        best_dev_acc = val_acc
        save_model(model, "best.pt")
        print(f"   🎯 发现更好的模型！验证准确率: {best_dev_acc:.3f}")

train_end_time = time.time()
training_time_sec = train_end_time - train_start_time
training_time_min = training_time_sec / 60

save_model(model, "last.pt")

print("\n✅ 训练完成！")
print(f"   - 最佳验证准确率: {best_dev_acc:.3f}")
print(f"   - 最佳模型已保存: {os.path.join(SAVE_PATH, 'best.pt')}")
print(f"   - 最后模型已保存: {os.path.join(SAVE_PATH, 'last.pt')}")

In [ ]:
"""
目的：在测试集上评估训练好的LSTM模型性能，使用最佳模型进行最终评估

数据流向：
输入：SAVE_PATH（模型保存路径）、best.pt（最佳模型文件）、test_loader（测试数据加载器）
-> 加载最佳模型 -> 评估模型
输出：测试集评估指标（损失、准确率、精确率、召回率、F1分数）

操作步骤：
1. 从保存路径加载最佳模型的参数：使用torch.load读取best.pt文件，load_state_dict加载到模型
2. 调用evaluate函数在测试集上评估模型，获得所有评估指标
3. 打印实验名称和测试集评估结果（损失、准确率、精确率、召回率、F1分数）
4. 打印训练时间（秒和分钟）
5. 打印最终测试准确率
"""

print("\n加载最佳模型进行测试集评估...")
model.load_state_dict(torch.load(os.path.join(SAVE_PATH, "best.pt")))

test_acc, test_precision, test_recall, test_f1, test_loss = evaluate(
    model, test_loader, criterion, device
)
print("\n实验名称:"+EXP_NAME)
print("\n测试集评估结果:")
print(f"  - Loss: {test_loss:.3f}")
print(f"  - Accuracy: {test_acc:.3f}")
print(f"  - Precision: {test_precision:.3f}")
print(f"  - Recall: {test_recall:.3f}")
print(f"  - F1 Score: {test_f1:.3f}")
print(f"   - 训练时间: {training_time_sec:.1f} 秒 ({training_time_min:.2f} 分钟)")

print(f"\n🎉 最终测试准确率: {test_acc:.3f}")